<div dir="rtl" lang="he" align="right" markdown="1">

# RAG מאפס

מודל שפה יודע רק את מה שראה באימון. הוא לא מכיר את המסמכים שלך, וכששואלים אותו עליהם הוא בדרך כלל ימציא תשובה שנשמעת סבירה לגמרי. הפתרון אינו לאמן אותו מחדש, אלא למצוא את המסמך הנכון ולהגיש לו אותו יחד עם השאלה. זה כל הרעיון של `RAG`.

החלק הקשה הוא המציאה. הפרק הזה בונה אותו שלב אחר שלב, ומודד כל שלב על 46 מסמכים ו-28 שאלות שהתשובה הנכונה שלהן ידועה מראש.

**מה נבנה כאן:** `pipeline` שלם של `retrieval`, חיתוך לחתיכות, `embeddings`, `BM25`, מיזוג של השניים, ולבסוף `reranker`, ובסוף טבלה אחת שמראה כמה פעמים כל שלב הביא את המסמך הנכון.

**מה זה דורש ממך:** כמעט כלום. אין כאן אף קריאה למודל שפה, אין `API key`, אין `GPU`. הכל רץ על המעבד של הלפטופ שלך, וכל המספרים בפרק הזה מחושבים אצלך, לא מצוטטים מבלוג.

</div>

In [ ]:
from pathlib import Path

from aihe import viz
from aihe.chunking import STRATEGIES, recursive
from aihe.embeddings import cross_encoder_scorer, encode
from aihe.pipeline import (
    build_chunks,
    contextualise,
    dense_retriever,
    evaluate,
    evaluate_by_kind,
    hybrid_retriever,
    load_corpus,
    reranked_retriever,
    sparse_retriever,
)
from aihe.retrieval import BM25, cosine_search, tokenize

CORPUS = Path("data/meridian.json") if Path("data/meridian.json").exists() \
    else Path("chapters/01-rag/data/meridian.json")
corpus = load_corpus(CORPUS)
len(corpus.documents), len(corpus.queries)

<div dir="rtl" lang="he" align="right" markdown="1">

## המסמכים והשאלות

הקורפוס כאן הוא תיעוד של `API` מומצא בשם `Meridian`. הוא מומצא בכוונה: כל מסמך נכתב כדי להכיל תופעה מסוימת שרצינו למדוד, ולכן אפשר לבדוק כל טענה בפרק במקום להאמין לה.

לכל שאלה יש רשימה של המסמכים שבאמת עונים עליה, והשאלות מחולקות לשלושה סוגים. `identifier` היא שאלה שמכילה מחרוזת מדויקת כמו קוד שגיאה. `paraphrase` שואלת על אותו דבר במילים אחרות לגמרי. `mixed` היא תערובת. החלוקה הזו היא מה שיגלה בהמשך איזה מנגנון עובד איפה.

</div>

In [ ]:
example = corpus.documents[1]
print(example["title"])
print(example["text"][:220], "...")

print("\nשאלות לדוגמה:")
for q in (corpus.queries[1], corpus.queries[6], corpus.queries[11]):
    print(f"  [{q['kind']:10s}] {q['question']:58s} -> {q['relevant']}")

<div dir="rtl" lang="he" align="right" markdown="1">

## שלב ראשון: לחתוך את המסמכים

מסמך שלם הוא יחידה גדולה מדי. אם נחפש לפי מסמך שלם, החלק הרלוונטי יטבע בתוך כל השאר. לכן חותכים כל מסמך לחתיכות, `chunks`, ומחפשים בהן.

יש שלוש דרכים לחתוך, והן נבדלות במה שהן מוכנות לשבור. החיתוך הקבוע פשוט סופר תווים ולא אכפת לו אם הוא חותך באמצע מילה. החיתוך הרקורסיבי מנסה קודם לשבור בין פסקאות, אחר כך בין שורות, אחר כך בין משפטים, ורק בלית ברירה באמצע מילה. החיתוך לפי מבנה הולך אחרי הפסקאות של המסמך עצמו.

שימו לב שכל `chunk` שומר את המיקום המדויק שלו במסמך המקורי. `chunk` שאי אפשר למצוא במקור הוא `chunk` שאי אפשר לנפות באגים בעזרתו.

</div>

In [ ]:
sample = corpus.documents[0]["text"]
for name in ("fixed", "recursive", "by_structure"):
    pieces = STRATEGIES[name](sample, size=200)
    print(f"{name:14s} {len(pieces):2d} chunks   first: {pieces[0].text[:64]!r}")

# the offsets always point back at the source
first = recursive(sample, size=200)[0]
print("\noffsets exact:", sample[first.start:first.end] == first.text)

In [ ]:
distributions = {
    name: [len(c) for d in corpus.documents for c in STRATEGIES[name](d["text"], size=200)]
    for name in ("fixed", "recursive", "by_structure")
}
viz.chunk_sizes(distributions, title="the same corpus, cut three ways");

<div dir="rtl" lang="he" align="right" markdown="1">

## שלב שני: להפוך טקסט למספרים

כדי לחפש לפי משמעות ולא לפי מילים, צריך להפוך טקסט למספרים. `embedding` הוא רשימה של מספרים שמייצגת את המשמעות של קטע טקסט. שני קטעים שמדברים על אותו דבר יקבלו רשימות דומות, גם אם אין להם אף מילה משותפת. זו בדיוק היכולת שחיפוש מילים רגיל לא יכול לתת.

הדמיון בין שני `embeddings` נמדד בזווית ביניהם ולא במרחק, וקטור ארוך וקטור קצר שמצביעים לאותו כיוון נחשבים זהים. המודל שאנחנו משתמשים בו כאן הוא קטן, רץ על מעבד רגיל, ותומך גם בעברית, מה שיהיה חשוב בפרק הבא.

</div>

In [ ]:
chunks = build_chunks(corpus, size=200, overlap=40)
bare_texts = [c.text for c in chunks]

chunk_vectors = encode(bare_texts)
question_vectors = encode(corpus.questions())
print(f"{len(chunks)} chunks -> {chunk_vectors.shape[1]} numbers each")

# what "similar meaning, different words" looks like
probe = encode(["how long is a token valid"])[0]
for rank, (i, score) in enumerate(cosine_search(probe, chunk_vectors, k=3), start=1):
    print(f"  {rank}. {score:.3f}  {chunks[i].text[:70]}")

<div dir="rtl" lang="he" align="right" markdown="1">

## המדידה הראשונה

עכשיו אפשר למדוד. השאלה היא פשוטה: מתוך 28 השאלות, בכמה מהן המסמך הנכון הגיע בשלושת הראשונים.

נשתמש בשני מספרים. `recall@3` הוא החלק מהמסמכים הנכונים שהגיעו לשלושת הראשונים, ו-`recall@1` הוא החלק מהשאלות שבהן המסמך הנכון היה ממש ראשון. נדווח גם על שיעור הכישלון, `1 − recall@3`, כי ירידה מ-`23%` ל-`11%` נקראת כמו השיפור שהיא, בעוד ש"עלייה מ-`77%` ל-`89%`" לא.

</div>

In [ ]:
plain = dense_retriever(question_vectors, chunk_vectors, chunks)
step1 = evaluate(plain, corpus, k=3)
print({k: round(v, 3) for k, v in step1.items()})
print("\nby question kind:", {k: round(v, 2) for k, v in evaluate_by_kind(plain, corpus, k=3).items()})

<div dir="rtl" lang="he" align="right" markdown="1">

## השורה אחת שמתקנת את רוב הבעיה

כשחותכים מסמך לחתיכות, כל חתיכה מאבדת את הדבר היחיד שאמר על מה המסמך. `chunk` מאמצע מסמך שכותרתו *Rejected credentials* כבר לא מכיל את המילה `credentials`, ולכן שאלה ששואלת למה השרת דוחה את הפרטים שלי לא מוצאת אותו, לא ב-`embedding` ולא ב-`BM25`.

התיקון הוא להחזיר את ההקשר: להדביק את כותרת המסמך בתחילת כל `chunk` שנחתך ממנו. זו הצורה הזולה ביותר של מה ש-Anthropic קוראת לו `contextual retrieval`, והיא שורה אחת של קוד.

</div>

In [ ]:
rich_texts = contextualise(chunks, corpus)
print("before:", bare_texts[3][:72])
print("after: ", rich_texts[3][:72])

rich_vectors = encode(rich_texts)
with_context = dense_retriever(question_vectors, rich_vectors, chunks)
step2 = evaluate(with_context, corpus, k=3)

print(f"\nfailed@3   {step1['failed@3']:.3f}  ->  {step2['failed@3']:.3f}")
print("paraphrase recall@3 "
      f"{evaluate_by_kind(plain, corpus, 3)['paraphrase']:.2f}  ->  "
      f"{evaluate_by_kind(with_context, corpus, 3)['paraphrase']:.2f}")

<div dir="rtl" lang="he" align="right" markdown="1">

## שלב שלישי: חיפוש מילים, דווקא

ה-`embeddings` מטשטשים, וזו התכונה שלהם, וזה בדיוק מה שהופך אותם לגרועים במחרוזות מדויקות. קוד שגיאה כמו `TS-999` אינו דומה במשמעות לשום דבר, הוא פשוט צריך להימצא.

השיטה `BM25` היא דירוג לפי מילים משנות התשעים, והיא עדיין מנצחת בדיוק שם. מסמך מקבל ניקוד גבוה יותר ככל שיש בו יותר ממילות השאילתה, מילים נדירות שוות יותר מנפוצות, חזרה על מילה נותנת פחות ופחות, ומסמך ארוך לא מקבל יתרון לא הוגן.

</div>

In [ ]:
bm25 = BM25([tokenize(t) for t in rich_texts])
sparse = sparse_retriever(bm25, chunks)

print("BM25 on an exact identifier:")
for q in corpus.queries[:1] + [corpus.queries[18]]:
    print(f"  {q['question']:12s} -> {sparse(0, q['question'])[:3]}   (wanted {q['relevant']})")

print("\nBM25 alone, overall:", {k: round(v, 3) for k, v in evaluate(sparse, corpus, k=3).items()})
print("by kind:", {k: round(v, 2) for k, v in evaluate_by_kind(sparse, corpus, k=3).items()})

<div dir="rtl" lang="he" align="right" markdown="1">

## למזג את שתי הרשימות

יש לנו שני מחפשים שנכשלים במקומות שונים. צריך למזג את הרשימות שלהם, והבעיה היא שאי אפשר להשוות ניקוד של `BM25` לניקוד של דמיון קוסינוס, הם לא באותן יחידות.

השיטה `Reciprocal Rank Fusion` פותרת את זה בכך שהיא מתעלמת מהניקוד לגמרי ומסתכלת רק על המקום ברשימה. מסמך שהגיע מקום ראשון באחת ומקום עשירי בשנייה מקבל `1/61 + 1/70 = 0.0307`. מסמך שהגיע מקום שלישי בשתיהן מקבל `2/63 = 0.0317`, ומנצח. הסכמה עקבית שווה יותר ממקום ראשון אחד ומוצלח.

</div>

In [ ]:
lucky, steady = 1/61 + 1/70, 2/63
print(f"rank 1 and rank 10 -> {lucky:.4f}")
print(f"rank 3 and rank  3 -> {steady:.4f}   <- wins")

hybrid = hybrid_retriever(question_vectors, rich_vectors, bm25, chunks)
step3 = evaluate(hybrid, corpus, k=3)
print(f"\nfailed@3  {step2['failed@3']:.3f} -> {step3['failed@3']:.3f}")
print(f"recall@1  {evaluate(with_context, corpus, 1)['recall@1']:.3f} -> "
      f"{evaluate(hybrid, corpus, 1)['recall@1']:.3f}")

<div dir="rtl" lang="he" align="right" markdown="1">

## שלב רביעי: לקרוא שוב, לאט

עד כה כל השוואה נעשתה בין שני וקטורים שחושבו בנפרד, בלי שאף אחד מהם ידע על קיומו של השני. `cross-encoder` עושה משהו אחר: הוא קורא את השאלה ואת המסמך יחד ונותן ציון לזוג. זה מדויק הרבה יותר, וזה גם איטי מדי מכדי להריץ על מיליון מסמכים.

לכן מריצים אותו אחרון ועל מעט: מביאים רשימה רחבה וזולה, ואז מדרגים מחדש רק את הראשונים. שימו לב שהוא יכול רק לסדר מחדש את מה שקיבל, הוא לעולם לא יציל מסמך שלא הגיע אליו מלכתחילה.

</div>

In [ ]:
full_text = {d["id"]: f"{d['title']}. {d['text']}" for d in corpus.documents}
reranked = reranked_retriever(hybrid, lambda d: full_text[d], cross_encoder_scorer(), shortlist=5)

print(f"recall@1  {evaluate(hybrid, corpus, 1)['recall@1']:.3f} -> "
      f"{evaluate(reranked, corpus, 1)['recall@1']:.3f}")
print(f"recall@3  {step3['recall@3']:.3f} -> {evaluate(reranked, corpus, 3)['recall@3']:.3f}")

<div dir="rtl" lang="he" align="right" markdown="1">

## המדרגות, בתמונה אחת

עכשיו אפשר לראות את כל ארבעת השלבים יחד. שימו לב שהעמודה של שיעור הכישלון והעמודה של `recall@1` מספרות שני סיפורים שונים, ושניהם נכונים.

</div>

In [ ]:
steps = {
    "1 plain": plain,
    "2 + context": with_context,
    "3 + BM25": hybrid,
    "4 + reranker": reranked,
}
table = {name: evaluate(r, corpus, k=3) for name, r in steps.items()}

print(f"{'setup':16s} {'recall@1':>9s} {'recall@3':>9s} {'failed@3':>9s}")
for name, r in steps.items():
    print(f"{name:16s} {evaluate(r, corpus, 1)['recall@1']:9.3f} "
          f"{table[name]['recall@3']:9.3f} {table[name]['failed@3']:9.3f}")

viz.staircase({n: s["failed@3"] for n, s in table.items()}, k=3,
              title="failed retrievals, step by step");

<div dir="rtl" lang="he" align="right" markdown="1">

## מה כל שלב באמת תיקן

זו הנקודה החשובה בפרק, והיא נעלמת אם מסתכלים רק על עמודה אחת.

הדבקת ההקשר תיקנה **החמצות**, מסמכים שפשוט לא הוחזרו בכלל. זה השלב שהוריד את שיעור הכישלון בחצי, והוא גם היחיד שעשה זאת.

מיזוג `BM25` ו-`reranker` תיקנו **סדר**, מסמכים שכן הוחזרו אבל לא היו ראשונים. הם כמעט לא זזו בשיעור הכישלון, והם אלה שהעלו את `recall@1`.

ולכן, אם היינו מודדים רק `recall@3`, היינו מסיקים ששני השלבים האחרונים מיותרים. אם היינו מודדים רק `recall@1`, היינו מפספסים שהדבקת ההקשר היא הניצחון הגדול בפרק. מדד אחד לא מספיק.

מספר אחרון שכדאי לזכור: `reranker` שקיבל רשימה רחבה מדי דווקא הזיק. ב-20 מועמדים `recall@3` ירד מ-`0.911` ל-`0.875`, כי יותר מועמדים פירושם יותר הזדמנויות לדחוף תשובה טובה למטה.

</div>

In [ ]:
first, last = table["1 plain"], table["4 + reranker"]
r1_first = evaluate(plain, corpus, 1)["recall@1"]
r1_last = evaluate(reranked, corpus, 1)["recall@1"]

print(f"failed@3 {first['failed@3']:.1%} -> {last['failed@3']:.1%}"
      f"   (cut {(first['failed@3'] - last['failed@3']) / first['failed@3']:.0%})")
print(f"recall@1 {r1_first:.1%} -> {r1_last:.1%}")